In [75]:
import requests
import pandas as pd
from collections import defaultdict

In [76]:
# Step 2: Fetch all the indicator available on adapta
url = "https://sistema.adaptabrasil.mcti.gov.br/api/hierarquia/adaptabrasil"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
else:
    raise Exception(f"Failed to fetch data: {response.status_code}")

df = pd.json_normalize(data)

In [68]:
df.to_csv('./data/adapta_indicators_ids.csv', index=False)

In [69]:
import ast

scenario_rows = []
seen = set()

for raw in df["scenarios"].dropna():
    # scenarios can arrive as list/dict already, or as a serialized string
    if isinstance(raw, list):
        items = raw
    elif isinstance(raw, dict):
        items = [raw]
    elif isinstance(raw, str):
        try:
            items = ast.literal_eval(raw)
        except Exception:
            continue
    else:
        continue

    if isinstance(items, dict):
        items = [items]
    if not isinstance(items, list):
        continue

    for item in items:
        if not isinstance(item, dict):
            continue

        label = str(item.get("label", "")).strip()
        description = str(item.get("description", "")).strip().replace("\n", " ")

        # Simple EN translations for known scenario descriptions
        description_en = description
        if "RCP4.5" in description:
            description_en = "The optimistic scenario refers to RCP4.5, which projects an additional 4.5 W/m2 of energy forcing and stabilization of GHG emissions before 2100."
        elif "RCP8.5" in description and "pessimista" in description.lower():
            description_en = "The pessimistic scenario refers to RCP8.5, characterized by an accelerated rate of emissions with no projected stabilization. This scenario projects an additional energy forcing of 8.5 W/m2."
        elif "SSP2-4.5" in description:
            description_en = "The SSP2-4.5 scenario is the intermediate GHG emissions pathway, where CO2 emissions remain around current levels until mid-century."
        elif "SSP5-8.5" in description:
            description_en = "The SSP5-8.5 scenario is the very high GHG emissions pathway, where CO2 emissions approximately double relative to current levels by 2050."
        elif "1,5°C" in description:
            description_en = "Increase in global average temperature of 1.5C above pre-industrial levels under the high-emissions greenhouse gas concentration scenario (RCP 8.5), projected to occur between 2011 and 2040."
        elif "2,0°C" in description:
            description_en = "Increase in global average temperature of 2.0C above pre-industrial levels under the high-emissions greenhouse gas concentration scenario (RCP 8.5), projected to occur between 2040 and 2070."

        row = {
            "scenario_value": item.get("value"),
            "label": label,
            "description": description,
            "description_en": description_en,
            "default_value": item.get("default_value"),
        }

        key = tuple(row.items())
        if key not in seen:
            seen.add(key)
            scenario_rows.append(row)

scenarios_df = pd.DataFrame(scenario_rows)
scenarios_df.to_csv("./data/adapta_scenarios_extracted.csv", index=False)
scenarios_df

,scenario_value,label,description,description_en,default_value
0,49,Otimista,O cenário SSP2-4.5 é o cenário de emissões de ...,The SSP2-4.5 scenario is the intermediate GHG ...,1
1,50,Pessimista,O cenário SSP5-8.5 é o cenário de emissões de ...,The SSP5-8.5 scenario is the very high GHG emi...,0
2,53,Otimista,"O cenário otimista refere-se ao RCP4.5, que pr...","The optimistic scenario refers to RCP4.5, whic...",1
3,54,Pessimista,"O cenário pessimista refere-se ao RCP8.5, send...","The pessimistic scenario refers to RCP8.5, cha...",0
4,42,SWL2.0,"Aumento da temperatura média global em 2,0°C a...",Increase in global average temperature of 2.0C...,1
5,31,Otimista,"O cenário otimista refere-se ao RCP4.5, que pr...","The optimistic scenario refers to RCP4.5, whic...",1
6,32,Pessimista,"O cenário pessimista refere-se ao RCP8.5, send...","The pessimistic scenario refers to RCP8.5, cha...",0
7,35,Otimista,"O cenário otimista refere-se ao RCP4.5, que pr...","The optimistic scenario refers to RCP4.5, whic...",1
8,36,Pessimista,"O cenário pessimista refere-se ao RCP8.5, send...","The pessimistic scenario refers to RCP8.5, cha...",0
9,40,Otimista,"O cenário otimista refere-se ao RCP4.5, que pr...","The optimistic scenario refers to RCP4.5, whic...",1


In [ ]:
# Minimal per-indicator downloader (starts at indicator 2).
import ast
import os
import time

BASE_URL = "https://sistema.adaptabrasil.mcti.gov.br/api"
SLEEP = 0.5
MIN_INDICATOR_ID = 2
OUT_DIR = "/Users/amandaeames/Documents/gitrepo/CityCatalyst-global-data/dataset-review/reviews/br-mcti/br-adaptabrasil/releases/v1/sample/indicators"
os.makedirs(OUT_DIR, exist_ok=True)

# Resume support: skip only combinations already covered.
COVERAGE_PATH = f"{OUT_DIR}/adapta_request_coverage.csv"
COMBINED_PATH = f"{OUT_DIR}/adapta_city_data.csv"

existing_coverage = []
processed_keys = set()
if os.path.exists(COVERAGE_PATH):
    existing_coverage = pd.read_csv(COVERAGE_PATH).to_dict("records")
    processed_keys = {
        (
            int(r["indicator_id"]),
            int(r["year"]),
            str(r["requested_scenario_id"]),
        )
        for r in existing_coverage
    }

# Requested fixed rule for indicator 2.
IND2_YEARS = [2020, 2030, 2050]
IND2_SCENARIOS = [27, 28, 31, 32, 35, 36, 40, 41, 42, 43, 44, 49, 50, 51, 52, 53, 54]
FULL_SCENARIOS = [27, 28, 31, 32, 35, 36, 40, 41, 42, 43, 44, 49, 50, 51, 52, 53, 54]

session = requests.Session()
session.headers.update({"User-Agent": "CityCatalyst-data-review/1.0"})

hierarchy = session.get(f"{BASE_URL}/hierarquia/adaptabrasil", timeout=30)
hierarchy.raise_for_status()
df = pd.json_normalize(hierarchy.json())
df = df[df["id"] >= MIN_INDICATOR_ID]

coverage = []
combined = []

for _, row in df.iterrows():
    indicator_id = int(row["id"])

    if indicator_id == 2:
        years = IND2_YEARS
        scenarios = IND2_SCENARIOS
    else:
        years = [
            int(float(y))
            for y in str(row.get("years") or "").replace("[", "").replace("]", "").split(",")
            if str(y).strip()
        ]
        if not years and row.get("years_description"):
            yd = row.get("years_description")
            yd = ast.literal_eval(yd) if isinstance(yd, str) and yd.strip() else yd
            if isinstance(yd, dict):
                yd = [yd]
            years = [int(float(x["year"])) for x in (yd or []) if isinstance(x, dict) and x.get("year") is not None]

        raw = row.get("scenarios")
        parsed = ast.literal_eval(raw) if isinstance(raw, str) and raw.strip() else (raw or [])
        if isinstance(parsed, dict):
            parsed = [parsed]
        scenarios = [int(x["value"]) for x in parsed if isinstance(x, dict) and x.get("value") is not None]

        # Fallback: if future years exist but metadata has no scenarios, use full known scenario list.
        if not scenarios and any(y > 2024 for y in years):
            scenarios = FULL_SCENARIOS

    parts = []
    for year in years:
        scenario_ids = [None] if (indicator_id == 2 and year == 2020) else (scenarios if (year > 2024 and scenarios) else [None])

        for sid in scenario_ids:
            sid_part = "null" if sid is None else sid
            key = (indicator_id, year, str(sid_part))
            if key in processed_keys:
                continue

            url = f"{BASE_URL}/mapa-dados/BR/municipio/{indicator_id}/{year}/{sid_part}/adaptabrasil"
            r = session.get(url, timeout=30)

            if r.status_code == 200:
                data = pd.json_normalize(r.json())
                data["indicator_id"] = indicator_id
                data["year"] = year
                data["requested_scenario_id"] = sid_part
                parts.append(data)
                rows = len(data)
            else:
                rows = 0

            coverage.append({
                "indicator_id": indicator_id,
                "year": year,
                "requested_scenario_id": sid_part,
                "status_code": r.status_code,
                "rows_returned": rows,
            })
            time.sleep(SLEEP)

    if parts:
        out = pd.concat(parts, ignore_index=True)
        out.to_csv(f"{OUT_DIR}/adapta_city_data_indicator_{indicator_id}.csv", index=False)
        combined.append(out)

all_coverage = existing_coverage + coverage
pd.DataFrame(all_coverage).to_csv(COVERAGE_PATH, index=False)

if os.path.exists(COMBINED_PATH):
    existing_combined = pd.read_csv(COMBINED_PATH)
    combined.insert(0, existing_combined)

if combined:
    all_df = pd.concat(combined, ignore_index=True)
    all_df.to_csv(COMBINED_PATH, index=False)
    all_df
else:
    print("No new responses fetched.")
